# Paper 4 — 03 · Causal localization via activation patching (H1c)

**The load-bearing experiment.** For matched parallel pairs where the model complies on RO and refuses on EN, patch the RO residual at each block with the cached EN residual; measure refusal restoration (judge-scored), swept over layers. H1c predicts the restoration peak sits in the **detection band**. Plus the four controls (EXPERIMENT_DESIGN §5.2).

**Output:** `results/<short>/activation_patching.json`.

In [ ]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [ ]:
import os, json, gc, sys, hashlib
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reuse Paper 2 judge harness + Paper 3 helpers; Paper 4 src/ ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(DRIVE_ROOT / "src"))        # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## Configuration

In [ ]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


## 1. Load gap-exhibiting pairs (nb01) + bands (nb02)

In [ ]:
out = CONTRAST_DIR / short
from behavioral import gap_exhibiting_pairs
par = {json.loads(l)['id']: json.loads(l) for l in (out / 'parallel.jsonl').read_text().splitlines() if l.strip()}
gap_ids = gap_exhibiting_pairs(out / 'behavioral_labels.jsonl')
pairs = [par[i] for i in gap_ids if i in par]
bands = json.loads((RESULTS_DIR / short / 'bands.json').read_text())
print(f'{len(pairs)} gap-exhibiting pairs | detection band {bands["detection"]} | execution band {bands["execution"]}')
assert len(pairs) >= 8, 'Too few gap pairs; add the bias subset (EXPERIMENT_LOG 2026-06-01).'

## 2. Load anchor + capture EN/RO residuals for the gap pairs

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from capture import capture_assistant_prefix
tok = AutoTokenizer.from_pretrained(ANCHOR); tok.padding_side='left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(ANCHOR, torch_dtype=torch.bfloat16, device_map='cuda').eval()
n_blocks = model.config.num_hidden_layers
en_acts = capture_assistant_prefix(model, tok, [p['text_en'] for p in pairs])
ro_acts = capture_assistant_prefix(model, tok, [p['text_ro'] for p in pairs])
en_prompts = [p['text_en'] for p in pairs]; ro_prompts = [p['text_ro'] for p in pairs]
ids = [p['id'] for p in pairs]
PATCH_MAX_NEW_TOKENS = 128   # enough to reveal refuse/comply; lower = faster

## 3. Judge label_fn (Paper 2 refusal judge; cached)

In [ ]:
from llm_judge import Judge
from behavioral import label_refusals
judge = Judge(model='openai/gpt-5-mini')
label_fn = lambda triples: {k: v['label'] for k, v in label_refusals(judge, triples).items()}

## 4. Main sweep: patch RO <- EN at every layer (H1c)

Restoration = fraction of gap pairs flipped comply->refuse by the patch.
H1c predicts the peak sits in the **detection** band.

In [ ]:
from patching import run_patch_sweep, bootstrap_ci, refusal_indicator_list
res_path = RESULTS_DIR / short / 'activation_patching.json'
layers = list(range(n_blocks))
main = run_patch_sweep(model, tok, en_acts, ro_prompts, ids, layers, label_fn,
                       max_new_tokens=PATCH_MAX_NEW_TOKENS)
per_layer = []
for l in layers:
    ind = refusal_indicator_list(main[l]); lo, hi = bootstrap_ci(ind)
    per_layer.append({'layer': l, 'restoration': main[l]['refusal_rate'], 'boot95': [lo, hi], 'n': main[l]['n']})
peak = max(per_layer, key=lambda d: d['restoration'])
peak_band = 'detection' if peak['layer'] in bands['detection'] else ('execution' if peak['layer'] in bands['execution'] else 'other')
print(f"peak restoration {peak['restoration']:.2f} at layer {peak['layer']} ({peak_band} band)")

## 5. Controls (EXPERIMENT_DESIGN §5.2) at representative layers

In [ ]:
import numpy as np
# C1 benign->benign: matched benign crosslingual pairs must NOT induce refusal.
xl = [json.loads(l) for l in (PAPER2_ROOT / 'benchmark' / 'expanded' / 'crosslingual.jsonl').read_text().splitlines() if l.strip()]
ben = [r for r in xl if r['id'].split('_')[1] == 'beni'][:len(pairs)]
ben_en_acts = capture_assistant_prefix(model, tok, [r['text_en'] for r in ben])
ctrl_layers = sorted({peak['layer'], bands['det_peak'], bands['exe_peak']})
C = {}
C['benign']   = run_patch_sweep(model, tok, ben_en_acts, [r['text_ro'] for r in ben], [r['id'] for r in ben], ctrl_layers, label_fn, max_new_tokens=PATCH_MAX_NEW_TOKENS)
C['mismatch'] = run_patch_sweep(model, tok, en_acts[np.roll(np.arange(len(pairs)),1)], ro_prompts, ids, ctrl_layers, label_fn, max_new_tokens=PATCH_MAX_NEW_TOKENS)
C['random']   = run_patch_sweep(model, tok, en_acts, ro_prompts, ids, ctrl_layers, label_fn, random_norm_match=True, max_new_tokens=PATCH_MAX_NEW_TOKENS)
C['reverse']  = run_patch_sweep(model, tok, ro_acts, en_prompts, ids, ctrl_layers, label_fn, max_new_tokens=PATCH_MAX_NEW_TOKENS)  # EN<-RO noising
controls = {k: {l: v[l]['refusal_rate'] for l in ctrl_layers} for k, v in C.items()}
print('controls @', ctrl_layers, ':', controls)

## 6. Save result

In [ ]:
result = {'anchor_model': ANCHOR, 'short': short, 'analysis': 'activation_patching',
          'n_pairs_gap_exhibiting': len(pairs), 'bands': bands,
          'per_layer': per_layer, 'peak_layer': peak['layer'], 'peak_band': peak_band,
          'control_layers': ctrl_layers, 'controls': controls,
          'patch_max_new_tokens': PATCH_MAX_NEW_TOKENS}
res_path.write_text(json.dumps(result, indent=2))
print('wrote', res_path)

## 7. Restoration curve (peak in detection band => H1c supported)

In [ ]:
import matplotlib.pyplot as plt
L = [p['layer'] for p in per_layer]; R = [p['restoration'] for p in per_layer]
lo = [p['boot95'][0] for p in per_layer]; hi = [p['boot95'][1] for p in per_layer]
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(L, R, marker='o', ms=3, label='RO<-EN restoration')
ax.fill_between(L, lo, hi, alpha=0.15)
for b in bands['detection']: ax.axvspan(b-0.5, b+0.5, color='C0', alpha=0.06)
for b in bands['execution']: ax.axvspan(b-0.5, b+0.5, color='C1', alpha=0.06)
for k, m in zip(['benign','mismatch','random','reverse'], ['x','^','v','d']):
    ax.scatter(ctrl_layers, [controls[k][l] for l in ctrl_layers], marker=m, label=f'ctrl:{k}')
ax.set_xlabel('patch layer'); ax.set_ylabel('refusal rate after patch'); ax.legend(fontsize=8)
ax.set_title(f'{short}: RO<-EN restoration (peak L{peak["layer"]}, {peak_band})')
fig.tight_layout(); fig.savefig(FIG_DIR / f'restoration_{short}.pdf'); plt.show()